In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib

# Load the dataset
df = pd.read_csv('data_core.csv')
df.columns = df.columns.str.strip()

# Prepare the data for the classifier
X = df[['Nitrogen', 'Potassium', 'Phosphorous', 'Crop Type']]
y = df['Fertilizer Name']

# One-hot encode the 'Crop Type'
X = pd.get_dummies(X, columns=['Crop Type'], drop_first=True)

# Train the classifier
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
fertilizer_model = RandomForestClassifier(n_estimators=100, random_state=42)
fertilizer_model.fit(X_train, y_train)

# Save the trained model and its columns
joblib.dump(fertilizer_model, 'fertilizer_recommender_model.joblib')
joblib.dump(X.columns, 'fertilizer_model_columns.joblib')

print("✅ Fertilizer recommender model has been trained and saved.")

✅ Fertilizer recommender model has been trained and saved.


In [5]:
import pandas as pd
import joblib

# --- Load All Models and Data at Startup ---
yield_model = joblib.load('crop_yield_model.joblib')
yield_model_columns = joblib.load('model_columns.joblib')

fertilizer_model = joblib.load('fertilizer_recommender_model.joblib')
fertilizer_model_columns = joblib.load('fertilizer_model_columns.joblib')

# Load soil data to find "optimal" levels for each crop
soil_df = pd.read_csv('data_core.csv')
soil_df.columns = soil_df.columns.str.strip()
soil_df.rename(columns={'Crop Type': 'Crop'}, inplace=True)
optimal_nutrients = soil_df.groupby('Crop')[['Nitrogen', 'Potassium', 'Phosphorous']].mean()


def get_yield_prediction(data):
    """Predicts yield based on a dictionary of inputs."""
    # ... (function is the same as before)
    input_df = pd.DataFrame(columns=yield_model_columns)
    input_df.loc[0] = 0
    for key, value in data.items():
        if key in input_df.columns: input_df[key] = value
    col_name = f"District_{data.get('district')}"
    if col_name in input_df.columns: input_df[col_name] = 1
    col_name = f"Crop_{data.get('crop')}"
    if col_name in input_df.columns: input_df[col_name] = 1
    col_name = f"Season_{data.get('season')}"
    if col_name in input_df.columns: input_df[col_name] = 1
    return yield_model.predict(input_df[yield_model_columns])[0]


def get_fertilizer_recommendation(data):
    """Recommends a fertilizer based on a dictionary of inputs."""
    # ... (function is the same as before)
    input_df = pd.DataFrame(columns=fertilizer_model_columns)
    input_df.loc[0] = 0
    for key, value in data.items():
        if key in input_df.columns: input_df[key] = value
    crop_col = f"Crop Type_{data.get('crop')}"
    if crop_col in input_df.columns: input_df[crop_col] = 1
    return fertilizer_model.predict(input_df[fertilizer_model_columns])[0]


# --- Main Simulation ---
if __name__ == '__main__':
    
    # --- Farmer's Inputs (Now including their actual soil test results) ---
    farmer_inputs = {
        'district': 'CUTTACK',
        'crop': 'Paddy',
        'season': 'Kharif',
        'area': 10.0,
        'avg_temp': 28.5,
        'total_precip': 1200,
        # The farmer enters these values from their Soil Health Card
        'Nitrogen': 15.0,   # Example: Farmer's soil is low in Nitrogen
        'Potassium': 3.5,
        'Phosphorous': 17.0
    }
    
    # --- Step 1: Get Baseline Prediction with ACTUAL soil data ---
    baseline_yield = get_yield_prediction(farmer_inputs)
    print("--- Baseline Scenario (Using Farmer's Actual Soil Data) ---")
    print(f"Predicted yield with your current soil: {baseline_yield:.2f} Tonnes per Hectare")

    # --- Step 2: Get Fertilizer Recommendation for the ACTUAL soil ---
    recommended_fertilizer = get_fertilizer_recommendation(farmer_inputs)
    print(f"Recommended Fertilizer based on your soil: {recommended_fertilizer}")

    # --- Step 3: Simulate Optimized Conditions ---
    # Find the ideal nutrient levels for this crop
    optimal_values = optimal_nutrients.loc[farmer_inputs['crop']]
    
    optimized_inputs = farmer_inputs.copy()
    # Update the inputs to reflect the optimal nutrient levels
    optimized_inputs['Nitrogen'] = optimal_values['Nitrogen']
    optimized_inputs['Potassium'] = optimal_values['Potassium']
    optimized_inputs['Phosphorous'] = optimal_values['Phosphorous']
    
    optimized_yield = get_yield_prediction(optimized_inputs)

    # --- Step 4: Show the Difference ---
    yield_increase_percent = ((optimized_yield - baseline_yield) / baseline_yield) * 100 if baseline_yield > 0 else 0
    
    print("\n--- Optimized Scenario (After Fertilization) ---")
    print(f"Predicted yield after applying '{recommended_fertilizer}' to reach optimal levels: {optimized_yield:.2f} Tonnes per Hectare")
    print("\n----------------------------------------------------")
    print(f"✅ ACTIONABLE INSIGHT:")
    print(f"By using the recommended fertilizer, you could potentially increase your yield by {yield_increase_percent:.0f}%.")
    print("----------------------------------------------------")

--- Baseline Scenario (Using Farmer's Actual Soil Data) ---
Predicted yield with your current soil: 0.38 Tonnes per Hectare
Recommended Fertilizer based on your soil: 17-17-17

--- Optimized Scenario (After Fertilization) ---
Predicted yield after applying '17-17-17' to reach optimal levels: 0.38 Tonnes per Hectare

----------------------------------------------------
✅ ACTIONABLE INSIGHT:
By using the recommended fertilizer, you could potentially increase your yield by 0%.
----------------------------------------------------


In [6]:
import pandas as pd
import joblib

# --- Load All Models and Data ---
yield_model = joblib.load('crop_yield_model.joblib')
yield_model_columns = joblib.load('model_columns.joblib')
fertilizer_model = joblib.load('fertilizer_recommender_model.joblib')
fertilizer_model_columns = joblib.load('fertilizer_model_columns.joblib')

# (Include the get_yield_prediction and get_fertilizer_recommendation functions from the previous script)
def get_yield_prediction(data):
    input_df = pd.DataFrame(columns=yield_model_columns)
    input_df.loc[0] = 0
    for key, value in data.items():
        if key in input_df.columns: input_df[key] = value
    col_name = f"District_{data.get('district')}"
    if col_name in input_df.columns: input_df[col_name] = 1
    col_name = f"Crop_{data.get('crop')}"
    if col_name in input_df.columns: input_df[col_name] = 1
    col_name = f"Season_{data.get('season')}"
    if col_name in input_df.columns: input_df[col_name] = 1
    return yield_model.predict(input_df[yield_model_columns])[0]

def get_fertilizer_recommendation(data):
    input_df = pd.DataFrame(columns=fertilizer_model_columns)
    input_df.loc[0] = 0
    for key, value in data.items():
        if key in input_df.columns: input_df[key] = value
    crop_col = f"Crop Type_{data.get('crop')}"
    if crop_col in input_df.columns: input_df[crop_col] = 1
    return fertilizer_model.predict(input_df[fertilizer_model_columns])[0]


if __name__ == '__main__':
    # --- Farmer's Inputs ---
    farmer_inputs = {
        'district': 'CUTTACK',
        'crop': 'Urad', # Farmer is planning to plant a pulse
        'season': 'Kharif',
        'area': 10.0,
        'avg_temp': 28.5,
        'total_precip': 1200,
        'Nitrogen': 15.0, 'Potassium': 3.5, 'Phosphorous': 17.0
    }
    
    print("--- Farmer's Plan Analysis ---")
    
    # --- Insight 1: Direct Soil Recommendation ---
    recommended_fertilizer = get_fertilizer_recommendation(farmer_inputs)
    print(f"✅ Soil Recommendation: Based on your soil and plan to plant '{farmer_inputs['crop']}', the recommended fertilizer is '{recommended_fertilizer}'.")

    # --- Insight 2: Yield Optimization via Crop Selection ---
    baseline_yield = get_yield_prediction(farmer_inputs)
    print(f"\n✅ Yield Prediction: Your predicted yield for '{farmer_inputs['crop']}' is {baseline_yield:.2f} Tonnes per Hectare.")

    # Simulate an alternative crop
    alternative_crop = 'Maize'
    alternative_inputs = farmer_inputs.copy()
    alternative_inputs['crop'] = alternative_crop
    
    alternative_yield = get_yield_prediction(alternative_inputs)
    
    yield_increase_percent = ((alternative_yield - baseline_yield) / baseline_yield) * 100 if baseline_yield > 0 else 0

    if yield_increase_percent > 10:
        print(f"\n✅ Optimization Insight: Consider planting '{alternative_crop}' instead.")
        print(f"Our model predicts it could increase your yield by approximately {yield_increase_percent:.0f}%.")

--- Farmer's Plan Analysis ---
✅ Soil Recommendation: Based on your soil and plan to plant 'Urad', the recommended fertilizer is '17-17-17'.

✅ Yield Prediction: Your predicted yield for 'Urad' is 0.32 Tonnes per Hectare.

✅ Optimization Insight: Consider planting 'Maize' instead.
Our model predicts it could increase your yield by approximately 487%.


In [9]:
import pandas as pd
import joblib
import requests
from datetime import datetime

# --- Load Models and Data at Startup (if you have them) ---
# yield_model = joblib.load('crop_yield_model.joblib')
# yield_model_columns = joblib.load('model_columns.joblib')

# --- Coordinates for districts ---
district_coords = {
    'ANUGUL': (20.83, 85.09), 'BALANGIR': (20.71, 83.48), 'BALESHWAR': (21.49, 86.92),
    'BARGARH': (21.33, 83.62), 'BHADRAK': (21.06, 86.50), 'BOUDH': (20.84, 84.32),
    'CUTTACK': (20.46, 85.88), 'DEOGARH': (21.53, 84.73), 'DHENKANAL': (20.66, 85.59),
    'GAJAPATI': (19.08, 84.08), 'GANJAM': (19.38, 85.05), 'JAGATSINGHAPUR': (20.26, 86.17),
    'JAJAPUR': (20.84, 86.33), 'JHARSUGUDA': (21.87, 84.03), 'KALAHANDI': (19.85, 83.21),
    'KANDHAMAL': (20.43, 84.23), 'KENDRAPARA': (20.50, 86.42), 'KENDUJHAR': (21.63, 85.58),
    'KHORDHA': (20.18, 85.62), 'KORAPUT': (18.82, 82.72), 'MALKANGIRI': (18.35, 81.89),
    'MAYURBHANJ': (21.93, 86.75), 'NABARANGPUR': (19.23, 82.55), 'NAYAGARH': (20.13, 85.10),
    'NUAPADA': (20.83, 82.66), 'PURI': (19.81, 85.83), 'RAYAGADA': (19.17, 83.42),
    'SAMBALPUR': (21.47, 83.97), 'SONEPUR': (20.85, 83.90), 'SUNDARGARH': (22.12, 84.03)
}


# --- Define Season Dates ---
season_dates = {
    'Kharif'     : ('06-01', '10-31'),
    'Rabi'       : ('11-01', '04-30'),
    'Summer'     : ('03-01', '06-30'),
    'Winter'     : ('12-01', '02-28'),
    'Autumn'     : ('09-01', '12-31'),
    'Whole Year' : ('01-01', '12-31')
}

def get_climate_normals(district, season):
    """
    Fetches 30-year average weather conditions from NASA POWER for a given location and season.
    """
    if district not in district_coords or season not in season_dates:
        return None, None

    lat, lon = district_coords[district]
    start_month_day, end_month_day = season_dates[season]
    
    start_year, end_year = 1991, 2020

    base_url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters": "T2M_MAX,T2M_MIN,PRECTOTCORR",
        "community": "AG",
        "longitude": lon, "latitude": lat,
        "start": f"{start_year}0101", "end": f"{end_year}1231",
        "format": "JSON"
    }
    
    print("Fetching 30-year climate data from NASA POWER... (This may take a moment)")
    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()
        
        # Process data into a DataFrame
        df = pd.DataFrame(data['properties']['parameter'])
        
        # --- THIS IS THE CORRECTED LINE ---
        # Convert the index from string 'YYYYMMDD' to a proper DatetimeIndex
        df.index = pd.to_datetime(df.index, format='%Y%m%d')
        
        # Filter for the specific months of the season
        start_month = int(start_month_day.split('-')[0])
        end_month = int(end_month_day.split('-')[0])
        
        if start_month <= end_month:
            season_df = df[(df.index.month >= start_month) & (df.index.month <= end_month)]
        else: # Handle seasons that span years (like Rabi/Winter)
            season_df = df[(df.index.month >= start_month) | (df.index.month <= end_month)]

        # Calculate the 30-year average statistics
        season_df['T2M_AVG'] = (season_df['T2M_MAX'] + season_df['T2M_MIN']) / 2
        avg_seasonal_temp = season_df['T2M_AVG'].mean()
        avg_seasonal_precip = season_df.groupby(season_df.index.year)['PRECTOTCORR'].sum().mean()
        
        return avg_seasonal_temp, avg_seasonal_precip
        
    except requests.exceptions.RequestException as e:
        print(f"NASA POWER API request failed: {e}")
        return None, None

# --- Example Usage ---
if __name__ == '__main__':
    farmer_district = 'CUTTACK'
    farmer_season = 'Kharif'
    
    avg_temp, total_precip = get_climate_normals(farmer_district, farmer_season)
    
    if avg_temp is not None:
        print(f"\nBased on 30-year climate data for {farmer_district} ({farmer_season} season):")
        print(f"  - Expected Average Temperature: {avg_temp:.2f}°C")
        print(f"  - Expected Total Rainfall: {total_precip:.2f} mm")
        # You would then pass these values to your prediction function
    else:
        print("Could not retrieve climate data.")

Fetching 30-year climate data from NASA POWER... (This may take a moment)

Based on 30-year climate data for CUTTACK (Kharif season):
  - Expected Average Temperature: 28.03°C
  - Expected Total Rainfall: 1298.96 mm


C:\Users\ankad\AppData\Local\Temp\ipykernel_7452\2233017875.py:79: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  season_df['T2M_AVG'] = (season_df['T2M_MAX'] + season_df['T2M_MIN']) / 2


In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import joblib

# --- Step 1: Load Your Final, Fully-Enriched Dataset ---
filename = 'odisha_final_data.csv'
print(f"Reading your final dataset: '{filename}'...")
final_df = pd.read_csv(filename)
print("Dataset loaded successfully.")

# --- Step 2: Final, Robust Data Cleaning ---
# Strip leading/trailing whitespace from all column names
final_df.columns = final_df.columns.str.strip()
print("Column names cleaned.")

# Ensure all key columns are numeric, converting any errors
numeric_cols = ['Area', 'Production', 'Yield', 'Avg_Temp', 'Total_Precipitation', 'Nitrogen', 'Potassium', 'Phosphorous']
for col in numeric_cols:
    final_df[col] = pd.to_numeric(final_df[col], errors='coerce')

# Drop any rows that have missing values after cleaning
final_df.dropna(subset=numeric_cols, inplace=True)
print(f"Cleaned dataset has {len(final_df)} rows.")

# --- Step 3: Feature Engineering and Selection ---
# One-hot encode the categorical variables to be used in the model
final_df_encoded = pd.get_dummies(final_df, columns=['District', 'Crop', 'Season'], drop_first=True)

# Define our features (X) and target (y)
# We drop columns that would cause data leakage (like Production) or are no longer needed
X = final_df_encoded.drop(columns=['State', 'Year', 'Production', 'Yield', 'Area Units', 'Production Units', 'Crop_Category'])
y = final_df_encoded['Yield']

# --- Step 4: Split the Data for Training and Testing ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nTraining the model on {len(X_train)} samples and testing on {len(X_test)} samples.")

# --- Step 5: Train the Final Random Forest Model ---
# n_jobs=-1 uses all available CPU cores to speed up training
rf_model_final = RandomForestRegressor(n_estimators=100, random_state=42, oob_score=True, n_jobs=-1)
rf_model_final.fit(X_train, y_train)
print("Model training complete.")

# --- Step 6: Evaluate the Model's Performance ---
y_pred = rf_model_final.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\n--- Final Model Evaluation Results ---")
print(f"R-squared (R²): {r2:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Out-of-Bag (OOB) Score: {rf_model_final.oob_score_:.4f}")

# --- Step 7: Identify the Most Important Factors ---
feature_importances = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model_final.feature_importances_
}).sort_values('importance', ascending=False)

print("\n--- Top 15 Most Important Factors for Predicting Crop Yield ---")
print(feature_importances.head(15))

# --- Step 8: Save the Final Model for the API ---
model_filename = 'crop_yield_model.joblib'
columns_filename = 'model_columns.joblib'

# Compress the model to keep the file size down
joblib.dump(rf_model_final, model_filename, compress=3)
joblib.dump(X.columns, columns_filename)

print(f"\n✅ Final model saved as '{model_filename}'")
print(f"✅ Model columns saved as '{columns_filename}'")

Reading your final dataset: 'odisha_final_data.csv'...
Dataset loaded successfully.
Column names cleaned.
Cleaned dataset has 13811 rows.

Training the model on 11048 samples and testing on 2763 samples.
Model training complete.

--- Final Model Evaluation Results ---
R-squared (R²): 0.9513
Root Mean Squared Error (RMSE): 2.7001
Out-of-Bag (OOB) Score: 0.9485

--- Top 15 Most Important Factors for Predicting Crop Yield ---
                    feature  importance
59           Crop_Sugarcane    0.928193
0                      Area    0.017782
1                  Avg_Temp    0.013881
2       Total_Precipitation    0.011304
24         District_KORAPUT    0.002901
34      District_SUNDARGARH    0.002708
8          District_BARGARH    0.002358
27     District_NABARANGPUR    0.002147
15          District_GANJAM    0.001757
17         District_JAJAPUR    0.001652
18      District_JHARSUGUDA    0.001520
19       District_KALAHANDI    0.001359
7        District_BALESHWAR    0.001146
16  District_

In [12]:
#FINAL

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import joblib

# --- Step 1: Define the Crop Name Mapping ---
# Maps the specific names from the Odisha dataset to the general categories
# in the soil dataset.
crop_mapping = {
    'arhar/tur': 'pulses', 'bajra': 'millets', 'castor seed': 'oil seeds',
    'coriander': 'pulses', 'cotton(lint)': 'cotton', 'cowpea(lobia)': 'pulses',
    'gram': 'pulses', 'groundnut': 'ground nuts', 'horse-gram': 'pulses',
    'jowar': 'millets', 'linseed': 'oil seeds', 'maize': 'maize',
    'masoor': 'pulses', 'moong(green gram)': 'pulses', 'niger seed': 'oil seeds',
    'other kharif pulses': 'pulses', 'other rabi pulses': 'pulses',
    'peas & beans (pulses)': 'pulses', 'ragi': 'millets', 'rapeseed &mustard': 'oil seeds',
    'rice': 'paddy', 'safflower': 'oil seeds', 'sesamum': 'oil seeds',
    'small millets': 'millets', 'soyabean': 'oil seeds', 'sugarcane': 'sugarcane',
    'sunflower': 'oil seeds', 'tobacco': 'tobacco', 'urad': 'pulses', 'wheat': 'wheat'
}

# --- Step 2: Load and Merge the Datasets ---
print("Loading and merging datasets...")
# Load datasets
odisha_df = pd.read_csv('odisha_weather_crop_data.csv')
soil_df = pd.read_csv('data_core.csv')

# Clean column names
odisha_df.columns = odisha_df.columns.str.strip()
soil_df.columns = soil_df.columns.str.strip()

# Prepare soil data: calculate average nutrients per category
soil_df.rename(columns={'Crop Type': 'Crop_Category'}, inplace=True)
avg_nutrients = soil_df.groupby('Crop_Category')[['Nitrogen', 'Potassium', 'Phosphorous']].mean().reset_index()

# Prepare Odisha data: apply the mapping
odisha_df['Crop_Category'] = odisha_df['Crop'].str.lower().map(crop_mapping)
avg_nutrients['Crop_Category'] = avg_nutrients['Crop_Category'].str.lower()

# Merge the two dataframes
final_df = pd.merge(odisha_df, avg_nutrients, on='Crop_Category', how='left')
print("Merge complete.")

# --- Step 3: Final Data Cleaning ---
final_df.dropna(inplace=True)
numeric_cols = ['Area', 'Production', 'Yield', 'Avg_Temp', 'Total_Precipitation', 'Nitrogen', 'Potassium', 'Phosphorous']
for col in numeric_cols:
    final_df[col] = pd.to_numeric(final_df[col], errors='coerce')
final_df.dropna(subset=numeric_cols, inplace=True)
print(f"Final cleaned dataset has {len(final_df)} rows.")

# --- Step 4: Feature Engineering ---
final_df_encoded = pd.get_dummies(final_df, columns=['District', 'Crop', 'Season'], drop_first=True)
X = final_df_encoded.drop(columns=['State', 'Year', 'Production', 'Yield', 'Area Units', 'Production Units', 'Crop_Category'])
y = final_df_encoded['Yield']

# --- Step 5: Train/Test Split ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nTraining model on {len(X_train)} samples...")

# --- Step 6: Train the Final Model ---
rf_model_final = RandomForestRegressor(n_estimators=100, random_state=42, oob_score=True, n_jobs=-1)
rf_model_final.fit(X_train, y_train)
print("Model training complete.")

# --- Step 7: Evaluate the Model ---
y_pred = rf_model_final.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("\n--- Final Model Evaluation Results ---")
print(f"R-squared (R²): {r2:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

# --- Step 8: Analyze Feature Importance ---
feature_importances = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model_final.feature_importances_
}).sort_values('importance', ascending=False)
print("\n--- Top 15 Most Important Factors for Predicting Crop Yield ---")
print(feature_importances.head(15))

# --- Step 9: Save the Final Model ---
model_filename = 'crop_yield_model.joblib'
columns_filename = 'model_columns.joblib'
joblib.dump(rf_model_final, model_filename, compress=3) # Compress to keep file size small
joblib.dump(X.columns, columns_filename)
print(f"\n✅ Final model and columns saved successfully!")

Loading and merging datasets...
Merge complete.
Final cleaned dataset has 13811 rows.

Training model on 11048 samples...
Model training complete.

--- Final Model Evaluation Results ---
R-squared (R²): 0.9513
Root Mean Squared Error (RMSE): 2.7001

--- Top 15 Most Important Factors for Predicting Crop Yield ---
                    feature  importance
59           Crop_Sugarcane    0.928193
0                      Area    0.017782
1                  Avg_Temp    0.013881
2       Total_Precipitation    0.011304
24         District_KORAPUT    0.002901
34      District_SUNDARGARH    0.002708
8          District_BARGARH    0.002358
27     District_NABARANGPUR    0.002147
15          District_GANJAM    0.001757
17         District_JAJAPUR    0.001652
18      District_JHARSUGUDA    0.001520
19       District_KALAHANDI    0.001359
7        District_BALESHWAR    0.001146
16  District_JAGATSINGHAPUR    0.000828
22       District_KENDUJHAR    0.000819

✅ Final model and columns saved successfully!

In [14]:
import pandas as pd
import joblib
import requests
from datetime import datetime

# --- Step 1: Load All Models and Supporting Data at Startup ---
print("Loading all necessary models and data...")
# Load the trained models and their required column lists
yield_model = joblib.load('crop_yield_model.joblib')
yield_model_columns = joblib.load('model_columns.joblib')

fertilizer_model = joblib.load('fertilizer_recommender_model.joblib')
fertilizer_model_columns = joblib.load('fertilizer_model_columns.joblib')

# Load the soil data to determine the "optimal" nutrient levels for each crop
soil_df = pd.read_csv('data_core.csv')
soil_df.columns = soil_df.columns.str.strip()
soil_df.rename(columns={'Crop Type': 'Crop'}, inplace=True)
optimal_nutrients = soil_df.groupby('Crop')[['Nitrogen', 'Potassium', 'Phosphorous']].mean()
print("✅ Models and data loaded successfully.")


# --- Step 2: Define All Necessary Functions ---

# (Includes the final, corrected versions of our functions)
def get_climate_normals(district, season):
    """Fetches 30-year average weather conditions from NASA POWER."""
    # --- Coordinates for all 30 districts in Odisha ---
    district_coords = {
        'ANUGUL': (20.83, 85.09), 
        'BALANGIR': (20.71, 83.48), 
        'BALESHWAR': (21.49, 86.92),
        'BARGARH': (21.33, 83.62), 
        'BHADRAK': (21.06, 86.50), 
        'BOUDH': (20.84, 84.32),
        'CUTTACK': (20.46, 85.88), 
        'DEOGARH': (21.53, 84.73), 
        'DHENKANAL': (20.66, 85.59),
        'GAJAPATI': (19.08, 84.08), 
        'GANJAM': (19.38, 85.05), 
        'JAGATSINGHAPUR': (20.26, 86.17),
        'JAJAPUR': (20.84, 86.33), 
        'JHARSUGUDA': (21.87, 84.03), 
        'KALAHANDI': (19.85, 83.21),
        'KANDHAMAL': (20.43, 84.23), 
        'KENDRAPARA': (20.50, 86.42), 
        'KENDUJHAR': (21.63, 85.58),
        'KHORDHA': (20.18, 85.62), 
        'KORAPUT': (18.82, 82.72), 
        'MALKANGIRI': (18.35, 81.89),
        'MAYURBHANJ': (21.93, 86.75), 
        'NABARANGPUR': (19.23, 82.55), 
        'NAYAGARH': (20.13, 85.10),
        'NUAPADA': (20.83, 82.66), 
        'PURI': (19.81, 85.83), 
        'RAYAGADA': (19.17, 83.42),
        'SAMBALPUR': (21.47, 83.97), 
        'SONEPUR': (20.85, 83.90), 
        'SUNDARGARH': (22.12, 84.03)
    }

    # --- Define typical start/end dates for all agricultural seasons in Odisha ---
    season_dates = {
        'Kharif'     : ('06-01', '10-31'),
        'Rabi'       : ('11-01', '04-30'),
        'Summer'     : ('03-01', '06-30'),
        'Winter'     : ('12-01', '02-28'),
        'Autumn'     : ('09-01', '12-31'),
        'Whole Year' : ('01-01', '12-31')
    }

    if district not in district_coords or season not in season_dates: return None, None
    lat, lon = district_coords[district]
    start_month_day, end_month_day = season_dates[season]
    start_year, end_year = 1991, 2020

    params = {
        "parameters": "T2M_MAX,T2M_MIN,PRECTOTCORR", "community": "AG",
        "longitude": lon, "latitude": lat, "start": f"{start_year}0101",
        "end": f"{end_year}1231", "format": "JSON"
    }
    print("Fetching 30-year climate data from NASA POWER... (This may take a moment)")
    try:
        response = requests.get("https://power.larc.nasa.gov/api/temporal/daily/point", params=params)
        response.raise_for_status()
        data = response.json()
        df = pd.DataFrame(data['properties']['parameter'])
        df.index = pd.to_datetime(df.index, format='%Y%m%d')
        
        start_month, end_month = int(start_month_day.split('-')[0]), int(end_month_day.split('-')[0])
        
        if start_month <= end_month:
            season_df = df[(df.index.month >= start_month) & (df.index.month <= end_month)].copy()
        else:
            season_df = df[(df.index.month >= start_month) | (df.index.month <= end_month)].copy()

        season_df['T2M_AVG'] = (season_df['T2M_MAX'] + season_df['T2M_MIN']) / 2
        avg_seasonal_temp = season_df['T2M_AVG'].mean()
        avg_seasonal_precip = season_df.groupby(season_df.index.year)['PRECTOTCORR'].sum().mean()
        return avg_seasonal_temp, avg_seasonal_precip
    except requests.exceptions.RequestException as e:
        print(f"NASA POWER API request failed: {e}")
        return None, None

def get_yield_prediction(data):
    """Predicts yield based on a dictionary of inputs."""
    input_df = pd.DataFrame(columns=yield_model_columns)
    input_df.loc[0] = 0
    for key, value in data.items():
        if key in input_df.columns: input_df[key] = value
    for col_type in ['District', 'Crop', 'Season']:
        col_name = f"{col_type}_{data.get(col_type.lower())}"
        if col_name in input_df.columns: input_df[col_name] = 1
    return yield_model.predict(input_df[yield_model_columns])[0]

def get_fertilizer_recommendation(data):
    """Recommends a fertilizer based on a dictionary of inputs."""
    input_df = pd.DataFrame(columns=fertilizer_model_columns)
    input_df.loc[0] = 0
    for key, value in data.items():
        if key in input_df.columns: input_df[key] = value
    crop_col = f"Crop Type_{data.get('crop')}"
    if crop_col in input_df.columns: input_df[crop_col] = 1
    return fertilizer_model.predict(input_df[fertilizer_model_columns])[0]


# --- Step 3: Run the Full Simulation ---
if __name__ == '__main__':
    
    # --- Farmer's Inputs ---
    # In a real app, this data would come from the frontend or chatbot
    farmer_district = 'CUTTACK'
    farmer_crop = 'Paddy'
    farmer_season = 'Kharif'
    farmer_area = 10.0
    # Farmer provides their actual soil test results from their Soil Health Card
    farmer_soil_N = 15.0
    farmer_soil_P = 17.0
    farmer_soil_K = 3.5

    print(f"\n--- Generating Analysis for a Farmer in {farmer_district} ---")

    # --- Get Climate Forecast ---
    avg_temp, total_precip = get_climate_normals(farmer_district, farmer_season)
    
    if avg_temp is not None:
        print(f"Based on 30-year climate data for the {farmer_season} season:")
        print(f"  - Expected Average Temperature: {avg_temp:.2f}°C")
        print(f"  - Expected Total Rainfall: {total_precip:.2f} mm")

        # --- Create Input Dictionary for Models ---
        current_inputs = {
            'district': farmer_district, 'crop': farmer_crop, 'season': farmer_season,
            'area': farmer_area, 'avg_temp': avg_temp, 'total_precip': total_precip,
            'Nitrogen': farmer_soil_N, 'Potassium': farmer_soil_K, 'Phosphorous': farmer_soil_P
        }

        # --- Insight 1: Baseline Yield Prediction ---
        baseline_yield = get_yield_prediction(current_inputs)
        print(f"\n✅ Baseline Yield Prediction:")
        print(f"   Based on your farm's data and the expected climate, your predicted yield is {baseline_yield:.2f} Tonnes per Hectare.")

        # --- Insight 2: Fertilizer Recommendation ---
        recommended_fertilizer = get_fertilizer_recommendation(current_inputs)
        print(f"\n✅ Soil Treatment Recommendation:")
        print(f"   To balance your soil for '{farmer_crop}', the recommended fertilizer is '{recommended_fertilizer}'.")

        # --- Insight 3: Optimization Simulation ---
        optimal_values = optimal_nutrients.loc[farmer_crop]
        optimized_inputs = current_inputs.copy()
        optimized_inputs['Nitrogen'] = optimal_values['Nitrogen']
        optimized_inputs['Potassium'] = optimal_values['Potassium']
        optimized_inputs['Phosphorous'] = optimal_values['Phosphorous']
        
        optimized_yield = get_yield_prediction(optimized_inputs)
        yield_increase = ((optimized_yield - baseline_yield) / baseline_yield) * 100 if baseline_yield > 0 else 0

        if yield_increase > 5: # Only show if the increase is meaningful
            print(f"\n✅ Optimization Insight:")
            print(f"   Our simulation shows that by applying the recommended fertilizer to reach optimal soil levels,")
            print(f"   you could potentially increase your yield to {optimized_yield:.2f} Tonnes per Hectare (a {yield_increase:.0f}% increase).")
    else:
        print("\nCould not retrieve climate data to make a prediction.")

Loading all necessary models and data...
✅ Models and data loaded successfully.

--- Generating Analysis for a Farmer in CUTTACK ---
Fetching 30-year climate data from NASA POWER... (This may take a moment)
Based on 30-year climate data for the Kharif season:
  - Expected Average Temperature: 28.03°C
  - Expected Total Rainfall: 1298.96 mm

✅ Baseline Yield Prediction:
   Based on your farm's data and the expected climate, your predicted yield is 0.58 Tonnes per Hectare.

✅ Soil Treatment Recommendation:
   To balance your soil for 'Paddy', the recommended fertilizer is '17-17-17'.

✅ Optimization Insight:
   Our simulation shows that by applying the recommended fertilizer to reach optimal soil levels,
   you could potentially increase your yield to 1.27 Tonnes per Hectare (a 119% increase).


In [15]:
import pandas as pd
import joblib

# (Include the get_yield_prediction and get_fertilizer_recommendation functions from the previous script)
def get_yield_prediction(data):
    """Predicts yield based on a dictionary of inputs."""
    # This function should load the yield_model and yield_model_columns
    model = joblib.load('crop_yield_model.joblib')
    model_columns = joblib.load('model_columns.joblib')
    input_df = pd.DataFrame(columns=model_columns)
    input_df.loc[0] = 0
    for key, value in data.items():
        if key in input_df.columns: input_df[key] = value
    col_name = f"District_{data.get('district')}"
    if col_name in input_df.columns: input_df[col_name] = 1
    col_name = f"Crop_{data.get('crop')}"
    if col_name in input_df.columns: input_df[col_name] = 1
    col_name = f"Season_{data.get('season')}"
    if col_name in input_df.columns: input_df[col_name] = 1
    return model.predict(input_df[model_columns])[0]

def get_fertilizer_recommendation(data):
    """Recommends a fertilizer based on a dictionary of inputs."""
    # This function should load the fertilizer_model and fertilizer_model_columns
    model = joblib.load('fertilizer_recommender_model.joblib')
    model_columns = joblib.load('fertilizer_model_columns.joblib')
    input_df = pd.DataFrame(columns=model_columns)
    input_df.loc[0] = 0
    for key, value in data.items():
        if key in input_df.columns: input_df[key] = value
    crop_col = f"Crop Type_{data.get('crop')}"
    if crop_col in input_df.columns: input_df[crop_col] = 1
    return model.predict(input_df[model_columns])[0]


if __name__ == '__main__':
    # --- Farmer's Inputs ---
    farmer_inputs = {
        'district': 'CUTTACK',
        'crop': 'Urad',
        'season': 'Kharif',
        'area': 10.0,
        'avg_temp': 28.5,
        'total_precip': 1200,
        'Nitrogen': 15.0, 
        'Potassium': 3.5, 
        'Phosphorous': 17.0
    }
    
    print(f"--- Analysis for Your Plan: Planting '{farmer_inputs['crop']}' in {farmer_inputs['district']} ---")

    # --- Insight 1: Soil Treatment Recommendation ---
    recommended_fertilizer = get_fertilizer_recommendation(farmer_inputs)
    print(f"\n✅ Soil Recommendation:")
    print(f"   Based on your soil and plan to plant '{farmer_inputs['crop']}', the recommended fertilizer is '{recommended_fertilizer}'.")

    # --- Insight 2: Yield Prediction & Risk Assessment ---
    baseline_yield = get_yield_prediction(farmer_inputs)
    
    # Simulate a drought scenario (e.g., 30% less rainfall)
    drought_inputs = farmer_inputs.copy()
    drought_inputs['total_precip'] *= 0.70 # 30% less rain
    drought_yield = get_yield_prediction(drought_inputs)
    
    print(f"\n✅ Yield Prediction & Risk Assessment:")
    print(f"   With normal rainfall, your predicted yield is {baseline_yield:.2f} Tonnes per Hectare.")
    print(f"   In a drought scenario (30% less rainfall), the predicted yield could drop to {drought_yield:.2f} Tonnes per Hectare.")

--- Analysis for Your Plan: Planting 'Urad' in CUTTACK ---

✅ Soil Recommendation:
   Based on your soil and plan to plant 'Urad', the recommended fertilizer is '17-17-17'.

✅ Yield Prediction & Risk Assessment:
   With normal rainfall, your predicted yield is 0.58 Tonnes per Hectare.
   In a drought scenario (30% less rainfall), the predicted yield could drop to 0.58 Tonnes per Hectare.


In [2]:
import pandas as pd
import joblib
import requests
from datetime import datetime

# --- Step 1: Load All Models and Supporting Data at Startup ---
print("Loading all necessary models and data...")
# Load the trained models and their required column lists
try:
    yield_model = joblib.load('crop_yield_model.joblib')
    yield_model_columns = joblib.load('model_columns.joblib')
    fertilizer_model = joblib.load('fertilizer_recommender_model.joblib')
    fertilizer_model_columns = joblib.load('fertilizer_model_columns.joblib')

    # Load the soil data to determine the "optimal" nutrient levels for each crop
    soil_df = pd.read_csv('data_core.csv')
    soil_df.columns = soil_df.columns.str.strip()
    soil_df.rename(columns={'Crop Type': 'Crop'}, inplace=True)
    optimal_nutrients = soil_df.groupby('Crop')[['Nitrogen', 'Potassium', 'Phosphorous']].mean()
    print("✅ Models and data loaded successfully.")
except FileNotFoundError as e:
    print(f"Error: A required file is missing. Please check that all .joblib and .csv files are present. Missing file: {e.filename}")
    exit()


# --- Step 2: Define All Necessary Functions ---
def get_climate_normals(district, season):
    # Dictionaries for coordinates and season dates
    # --- Coordinates for all 30 districts in Odisha ---
    district_coords = {
        'ANUGUL': (20.83, 85.09), 
        'BALANGIR': (20.71, 83.48), 
        'BALESHWAR': (21.49, 86.92),
        'BARGARH': (21.33, 83.62), 
        'BHADRAK': (21.06, 86.50), 
        'BOUDH': (20.84, 84.32),
        'CUTTACK': (20.46, 85.88), 
        'DEOGARH': (21.53, 84.73), 
        'DHENKANAL': (20.66, 85.59),
        'GAJAPATI': (19.08, 84.08), 
        'GANJAM': (19.38, 85.05), 
        'JAGATSINGHAPUR': (20.26, 86.17),
        'JAJAPUR': (20.84, 86.33), 
        'JHARSUGUDA': (21.87, 84.03), 
        'KALAHANDI': (19.85, 83.21),
        'KANDHAMAL': (20.43, 84.23), 
        'KENDRAPARA': (20.50, 86.42), 
        'KENDUJHAR': (21.63, 85.58),
        'KHORDHA': (20.18, 85.62), 
        'KORAPUT': (18.82, 82.72), 
        'MALKANGIRI': (18.35, 81.89),
        'MAYURBHANJ': (21.93, 86.75), 
        'NABARANGPUR': (19.23, 82.55), 
        'NAYAGARH': (20.13, 85.10),
        'NUAPADA': (20.83, 82.66), 
        'PURI': (19.81, 85.83), 
        'RAYAGADA': (19.17, 83.42),
        'SAMBALPUR': (21.47, 83.97), 
        'SONEPUR': (20.85, 83.90), 
        'SUNDARGARH': (22.12, 84.03)
    }

    # --- Define typical start/end dates for all agricultural seasons in Odisha ---
    season_dates = {
        'Kharif'     : ('06-01', '10-31'),
        'Rabi'       : ('11-01', '04-30'),
        'Summer'     : ('03-01', '06-30'),
        'Winter'     : ('12-01', '02-28'),
        'Autumn'     : ('09-01', '12-31'),
        'Whole Year' : ('01-01', '12-31')
    }

    if district not in district_coords or season not in season_dates: return None, None
    lat, lon = district_coords[district]
    start_month_day, end_month_day = season_dates[season]
    start_year, end_year = 1991, 2020

    params = {
        "parameters": "T2M_MAX,T2M_MIN,PRECTOTCORR", "community": "AG", "longitude": lon,
        "latitude": lat, "start": f"{start_year}0101", "end": f"{end_year}1231", "format": "JSON"
    }
    print("\nFetching 30-year climate data from NASA POWER...")
    try:
        response = requests.get("https://power.larc.nasa.gov/api/temporal/daily/point", params=params)
        response.raise_for_status()
        data = response.json()
        df = pd.DataFrame(data['properties']['parameter'])
        df.index = pd.to_datetime(df.index, format='%Y%m%d')
        
        start_month, end_month = int(start_month_day.split('-')[0]), int(end_month_day.split('-')[0])
        
        if start_month <= end_month:
            season_df = df[(df.index.month >= start_month) & (df.index.month <= end_month)].copy()
        else:
            season_df = df[(df.index.month >= start_month) | (df.index.month <= end_month)].copy()

        season_df['T2M_AVG'] = (season_df['T2M_MAX'] + season_df['T2M_MIN']) / 2
        avg_seasonal_temp = season_df['T2M_AVG'].mean()
        avg_seasonal_precip = season_df.groupby(season_df.index.year)['PRECTOTCORR'].sum().mean()
        return avg_seasonal_temp, avg_seasonal_precip
    except requests.exceptions.RequestException as e:
        print(f"NASA POWER API request failed: {e}")
        return None, None

def get_yield_prediction(data):
    input_df = pd.DataFrame(columns=yield_model_columns)
    input_df.loc[0] = 0
    for key, value in data.items():
        if key in input_df.columns: input_df[key] = value
    for col_type in ['District', 'Crop', 'Season']:
        col_name = f"{col_type}_{data.get(col_type.lower())}"
        if col_name in input_df.columns: input_df[col_name] = 1
    return yield_model.predict(input_df[yield_model_columns])[0]

def get_fertilizer_recommendation(data):
    input_df = pd.DataFrame(columns=fertilizer_model_columns)
    input_df.loc[0] = 0
    for key, value in data.items():
        if key in input_df.columns: input_df[key] = value
    crop_col = f"Crop Type_{data.get('crop')}"
    if crop_col in input_df.columns: input_df[crop_col] = 1
    return fertilizer_model.predict(input_df[fertilizer_model_columns])[0]


# --- Step 3: Run the Full Analysis ---
if __name__ == '__main__':
    
    # ---  INPUTS ---
    # This section contains the dummy inputs for a farmer's scenario.
    # In your final app, these would come from the frontend or chatbot.
    farmer_inputs = {
        'district': 'BARGARH',
        'crop': 'Rice', # Note: Our yield model uses 'Rice', but the fertilizer model uses 'Paddy'.
        'season': 'Kharif',
        'area': 10.0,
        'avg_temp': 0, # This will be replaced by the climate data
        'total_precip': 0, # This will be replaced by the climate data
        # The farmer provides their actual soil test results from their Soil Health Card
        'Nitrogen': 20.0,
        'Potassium': 10.0,
        'Phosphorous': 15.0
    }

    print(f"\n--- Generating Analysis for a Farmer in {farmer_inputs['district']} ---")

    # --- ANALYSIS ---
    # Get the long-term climate forecast for the location and season
    climate_temp, climate_precip = get_climate_normals(farmer_inputs['district'], farmer_inputs['season'])
    
    if climate_temp is not None:
        # Update the inputs with the fetched climate data
        farmer_inputs['avg_temp'] = climate_temp
        farmer_inputs['total_precip'] = climate_precip

        # Get the baseline yield prediction
        baseline_yield = get_yield_prediction(farmer_inputs)
        
        # Get the fertilizer recommendation
        # The fertilizer model was trained on 'Paddy', so we map 'Rice' to 'Paddy' for this function
        fertilizer_input = farmer_inputs.copy()
        if fertilizer_input['crop'] == 'Rice':
            fertilizer_input['crop'] = 'Paddy'
        recommended_fertilizer = get_fertilizer_recommendation(fertilizer_input)
        
        # Simulate a drought scenario (30% less rainfall)
        drought_inputs = farmer_inputs.copy()
        drought_inputs['total_precip'] *= 0.70
        drought_yield = get_yield_prediction(drought_inputs)
        
        # --- OUTPUT ---
        # Display the final, formatted analysis report
        print("\n" + "="*50)
        print("          FASAL SAHAYAK AI - CROP ANALYSIS REPORT")
        print("="*50)
        print(f"Location: {farmer_inputs['district']}, Odisha")
        print(f"Crop: {farmer_inputs['crop']} | Season: {farmer_inputs['season']}")
        print(f"Farm Area: {farmer_inputs['area']} Hectares")
        print("-"*50)
        print("\n✅ CLIMATE FORECAST (Based on 30-Year Averages):")
        print(f"   - Expected Average Temperature: {climate_temp:.2f}°C")
        print(f"   - Expected Total Rainfall: {climate_precip:.2f} mm")

        print("\n✅ YIELD PREDICTION:")
        print(f"   Based on your farm's data and the expected climate, your predicted yield is **{baseline_yield:.2f} Tonnes per Hectare**.")

        print("\n✅ SOIL TREATMENT RECOMMENDATION:")
        print(f"   To balance your soil for '{farmer_inputs['crop']}', the recommended fertilizer type is **'{recommended_fertilizer}'**.")

        print("\n✅ CLIMATE RISK ASSESSMENT:")
        print(f"   In a potential drought scenario (30% less rainfall), the predicted yield could drop to **{drought_yield:.2f} Tonnes per Hectare**.")
        print("="*50)
        
    else:
        print("\nCould not retrieve climate data to generate a full analysis.")

Loading all necessary models and data...
✅ Models and data loaded successfully.

--- Generating Analysis for a Farmer in BARGARH ---

Fetching 30-year climate data from NASA POWER...

          FASAL SAHAYAK AI - CROP ANALYSIS REPORT
Location: BARGARH, Odisha
Crop: Rice | Season: Kharif
Farm Area: 10.0 Hectares
--------------------------------------------------

✅ CLIMATE FORECAST (Based on 30-Year Averages):
   - Expected Average Temperature: 27.82°C
   - Expected Total Rainfall: 1261.12 mm

✅ YIELD PREDICTION:
   Based on your farm's data and the expected climate, your predicted yield is **1.35 Tonnes per Hectare**.

✅ SOIL TREATMENT RECOMMENDATION:
   To balance your soil for 'Rice', the recommended fertilizer type is **'DAP'**.

✅ CLIMATE RISK ASSESSMENT:
   In a potential drought scenario (30% less rainfall), the predicted yield could drop to **1.35 Tonnes per Hectare**.
